# Spend DNA

| Field | Value |
|---|---|
| **Name** | Patel Mohitkumar Dipakkumar |
| **Project** | Spend — Minor Project |
| **Academy** | The Unlox Academy |
| **Domailn** | Data Science |
| **Date** | 30 June 2026 |

# Importing(reading) File

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('rahul_transactions.csv')
print("Raw data shape:", df.shape)
print(df.head(3))
#df

Raw data shape: (1328, 8)
         Date   Time                       Description   Type  Amount  \
0  2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1   01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2   01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   

    Balance  Mode        Ref  
0  678275.0   UPI  TXN190872  
1  681007.0   UPI  TXN143064  
2  484728.0  NEFT  TXN246316  


# Feacture 1 : Transaction Parser

In [46]:
# FEATURE 1: TRANSACTION PARSER

# fixing date
df['date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)

#fixing column
df['amount'] = (df['Amount']
    .str.replace('₹', '', regex=False)      # ₹ hatao
    .str.replace('Rs.', '', regex=False)    # Rs. hatao
    .str.replace(',', '', regex=False)      # comma hatao (1,200 → 1200)
    .str.strip()                            # spaces hatao
)
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

# Fixing Type Column
df['type'] = df['Type'].str.lower().str.strip()
df['type'] = df['type'].replace({'dr': 'debit', 'cr': 'credit'})

# Removing Duplicates
before = len(df)
df = df.drop_duplicates()
after = len(df)

df = df.dropna(subset=['date', 'amount'])

print(f" Parsed {after} transactions across 6 months.")
print(f" Dropped {before - after} duplicates.")
print(f" Date column type: {df['date'].dtype}")
print(f" Amount column type: {df['amount'].dtype}")
print(f" Type values: {df['type'].unique()}")

 Parsed 1310 transactions across 6 months.
 Dropped 0 duplicates.
 Date column type: datetime64[ns]
 Amount column type: float64
 Type values: ['debit' 'credit']


In [47]:
# FEATURE 1: TRANSACTION PARSER
from datetime import datetime

df = pd.read_csv('rahul_transactions.csv')
print("Step 0 - Raw rows:", len(df))

def parse_date(date_str):
    formats = [
        '%Y-%m-%d',    # 2024-01-01
        '%d-%b-%y',    # 01-Jan-24
        '%d/%m/%y',    # 01/01/24
        '%d %b %Y',    # 01 Jan 2024
        '%d-%m-%Y',    # 01-01-2024
        '%d/%m/%Y',    # 01/01/2024
    ]
    for fmt in formats:
        try:
            return datetime.strptime(str(date_str).strip(), fmt)
        except:
            continue
    return None

df['date'] = df['Date'].apply(parse_date)
print("Step 1 - NaT dates:", df['date'].isna().sum())

# amount fix
df['amount'] = (df['Amount']
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip())
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
print("Step 2 - NaN amounts:", df['amount'].isna().sum())

# Type fix
df['type'] = df['Type'].str.lower().str.strip()
df['type'] = df['type'].replace({'dr': 'debit', 'cr': 'credit'})
print("Step 3 - Type values:", df['type'].unique())

before = len(df)
df = df.drop_duplicates()
print(f"Step 4 - Dropped {before - len(df)} duplicates")

df = df.dropna(subset=['amount'])

print(f"\n✅ Parsed {len(df)} transactions across 6 months")
print(f"✅ Dropped {1328 - len(df)} duplicates")
print(f"✅ Date dtype: {df['date'].dtype}")
print(f"✅ Amount dtype: {df['amount'].dtype}")

Step 0 - Raw rows: 1328
Step 1 - NaT dates: 0
Step 2 - NaN amounts: 0
Step 3 - Type values: ['debit' 'credit']
Step 4 - Dropped 18 duplicates

✅ Parsed 1310 transactions across 6 months
✅ Dropped 18 duplicates
✅ Date dtype: datetime64[ns]
✅ Amount dtype: float64


# FEATURE 2: VENDOR EXTRACTOR

In [48]:
# FEATURE 2: VENDOR EXTRACTOR

print("=== UNIQUE DESCRIPTIONS (first 30) ===")
print(df['Description'].unique()[:30])

=== UNIQUE DESCRIPTIONS (first 30) ===
['AMAZON SELLER SVCS' 'BHIM-BMTC' 'NEFT-TECHCRUSH LABS-SALARY MAY24'
 'UPI-AMAN-8934@OKAXIS' 'BHIM-BLINKIT' 'BHIM ZEPTO'
 'UPI-UBER-2426@HDFCBANK' 'POS SWIGGY BANGALORE' 'UPI-GROWWPAY@HDFCBANK'
 'OLA ELECTRIC' 'BMS MOVIE TICKETS' 'POS OLA-PRIME' 'SWIGGY-INSTAMART'
 'UPI-STARBUCKS@AXIS' 'UPI-THIRDWAVE@OKAXIS' 'ANI Technologies'
 'BMTC BUS PASS' 'POS TRUFFLES' 'FLIPKART INDIA' 'POS SWIGGY-RESTAURANT'
 'GROFERS INDIA P L' 'POS UBER BANGALORE' 'BANGALORE ELEC SUPPLY'
 'TWC INDIA' 'UPI-BESCOM-BILL@HDFCBANK' 'UPI-AMAN-0816@OKAXIS'
 'ROPPEN TRANSPORTATION' 'OLA CABS' 'POS ZOMATO' 'UPI-AMAZONPAY@HDFCBANK']


In [49]:
# FEATURE 2: VENDOR EXTRACTOR
# such a crazy : it consumes most of my tiime to build this ..
# i got 17 uncatagorised value - so used AI to solve it..

vendor_map = {
    # FOOD DELIVERY
    'Swiggy': ['SWIGGY', 'BUNDL', 'Swiggy'],
    'Zomato': ['ZOMATO', 'ZOMATO MEDIA', 'ZOMATO-DINING'],

    # QUICK COMMERCE
    'Zepto':            ['ZEPTO', 'KIRANAKART'],
    'Blinkit':          ['BLINKIT','GROFERS'],
    'Swiggy Instamart': ['INSTAMART'],
    'BigBasket':        ['BIGBASKET', 'BIG BASKET'],

    # E-COMMERCE
    'Amazon':   ['AMAZON IN', 'AMZN', 'AMAZONIN',],
    'Amazon Pay':   ['Amazon Pay India'],
    'Amazon Prime': ['AMZN PRIME', 'PRIMEVIDEO', 'PRIME VIDEO'],
    'Flipkart': ['FLIPKART INDIA', 'UPI-FLIPKART', 'POS FLIPKART', 'FLIPKART INDIA'],
    'Flipkart Internet': ['FKART INTRNET', 'Flipkart Internet'],
    'Myntra':   ['MYNTRA'],
    'Nykaa':    ['NYKAA', 'INNOVATIVE RETAIL','FSN E-COMMERCE'],
    'Amazon Pay':   ['AMAZONPAY', 'AMAZON PAY'],

    # TRANSPORT
    'Uber':   ['UBER', 'Uber'],
    'Ola':    ['OLA', 'ANI Technologies'],
    'Rapido': ['RAPIDO', 'ROPPEN'],
    'BMTC':   ['BMTC', 'TUMMOC'],

    # CAFE
    'Starbucks':  ['STARBUCKS'],
    'CCD':        ['CCD', 'COFFEE DAY'],
    'Third Wave': ['THIRDWAVE', 'THIRD WAVE', 'TWC INDIA'],

    # RESTAURANTS
    'Restaurant':       ['RESTAURANT', 'EMPIRE', 'TRUFFLES',
                         'DINEOUT', 'MEGHANA'],
    'Dominos':    ['DOMINOS', 'JUBILANT'],
    'McDonalds':  ['MCDONALDS', 'MCDONALD'],
    'KFC':        ['KFC'],

    # SUBSCRIPTIONS
    'Netflix':      ['NETFLIX'],
    'Hotstar':      ['HOTSTAR','STAR INDIA'],
    'Spotify':      ['SPOTIFY'],
    'Amazon Prime': ['PRIMEVIDEO', 'PRIME VIDEO'],

    # ENTERTAINMENT
    'BookMyShow': ['BOOKMYSHOW', 'BMS','BIGTREE'],

    # UTILITIES
    'Electricity': ['BANGALORE ELEC', 'BESCOM'],
    'Mobile Bill': ['VI-RECHARGE', 'UPI-VI', 'AIRTEL', 'JIO','VI POSTPAID', 'VODAFONE'],
    'Petrol': ['PETROL', 'HP PETRO', 'HPCL', 'BPCL', 'INDIAN OIL', 'IOC', 'INDIAN'],
    'Water Bill':  ['BWSSB', 'WATER BILL'],

    # GROCERIES
    'DMart': ['DMART', 'AVENUE SUPERMARTS'],

    # INVESTMENTS
    'Zerodha': ['ZERODHA', 'ZERODHA-COIN'],
    'Groww':   ['GROWW', 'GROWWPAY'],

    # RENT & SALARY
    'Rent':   ['RENT', 'LANDLORD'],
    'Salary': ['SALARY', 'TECHCRUSH'],

    # P2P TRANSFERS
   'P2P Transfer': ['UPI-PRIYA', 'UPI-ANKIT', 'UPI-AMAN',
                 'UPI-VIKAS', 'UPI-SNEHA', 'UPI-KARAN', 'UPI-NEHA'],

    # ATM
    'Cash Withdrawal': ['ATM-WDL'],
}


# Functions
def extract_vendor(description):
    desc = str(description).upper()

    for vendor, keywords in vendor_map.items():
        for keyword in keywords:
            if keyword.upper() in desc:
                return vendor

    return 'Uncategorised'

df['vendor_clean'] = df['Description'].apply(extract_vendor)

print("=== TOP VENDORS ===")
print(df['vendor_clean'].value_counts().head(20))
print(f"\nTotal unique vendors: {df['vendor_clean'].nunique()}")

uncategorised = df[df['vendor_clean'] == 'Uncategorised']
print(f"\nUncategorised: {len(uncategorised)}")
if len(uncategorised) > 0:
    print("Match nahi hue descriptions:")
    print(uncategorised['Description'].unique())

=== TOP VENDORS ===
vendor_clean
Swiggy         223
Zomato         121
Ola             87
Restaurant      73
Uber            71
Zepto           71
Blinkit         55
Rapido          55
Starbucks       42
Amazon          41
Flipkart        39
BMTC            37
Third Wave      31
Amazon Pay      31
Petrol          28
Nykaa           27
CCD             26
Mobile Bill     24
DMart           22
Myntra          20
Name: count, dtype: int64

Total unique vendors: 37

Uncategorised: 10
Match nahi hue descriptions:
['AMAZON SELLER SVCS' 'UPI-AMAZON-PRIME@HDFCBANK']


In [50]:
print(df['vendor_clean'].unique())

['Uncategorised' 'BMTC' 'Salary' 'P2P Transfer' 'Blinkit' 'Zepto' 'Uber'
 'Swiggy' 'Groww' 'Ola' 'BookMyShow' 'Starbucks' 'Third Wave' 'Restaurant'
 'Flipkart' 'Electricity' 'Rapido' 'Zomato' 'Amazon Pay' 'Rent'
 'Mobile Bill' 'CCD' 'Swiggy Instamart' 'DMart' 'Petrol' 'Zerodha'
 'Myntra' 'Water Bill' 'Amazon' 'Netflix' 'Nykaa' 'Spotify' 'Hotstar'
 'Cash Withdrawal' 'Flipkart Internet' 'BigBasket' 'Amazon Prime']


# FEATURE 3: CATEGORY TAGGER

In [51]:
# FEATURE 3: CATEGORY TAGGER

category_map = {
    'Swiggy':            'Food Delivery',
    'Zomato':            'Food Delivery',
    'Zepto':             'Quick Commerce',
    'Blinkit':           'Quick Commerce',
    'Swiggy Instamart':  'Quick Commerce',
    'BigBasket':         'Quick Commerce',
    'Amazon':            'E-commerce',
    'Flipkart':          'E-commerce',
    'Myntra':            'E-commerce',
    'Nykaa':             'E-commerce',
    'Amazon Prime':      'Subscriptions',
    'Amazon Pay':        'Personal Transfer',
    'Flipkart Internet': 'Utilities',
    'Uber':              'Transport',
    'Ola':               'Transport',
    'Rapido':            'Transport',
    'BMTC':              'Transport',
    'Starbucks':         'Cafe',
    'CCD':               'Cafe',
    'Third Wave':        'Cafe',
    'Restaurant':        'Restaurants',
    'Netflix':           'Subscriptions',
    'Hotstar':           'Subscriptions',
    'Spotify':           'Subscriptions',
    'BookMyShow':        'Entertainment',
    'Electricity':       'Utilities',
    'Water Bill':        'Utilities',
    'Mobile Bill':       'Utilities',
    'Internet':          'Utilities',
    'Rent':              'Utilities',
    'Petrol':            'Fuel',
    'DMart':             'Groceries',
    'Zerodha':           'Investments',
    'Groww':             'Investments',
    'Salary':            'Income',
    'P2P Transfer':      'Personal Transfer',
    'Cash Withdrawal':   'Cash Withdrawal',
}

df['category'] = df['vendor_clean'].map(category_map)
df['category'] = df['category'].fillna('Uncategorised')

print(df['category'].apply(type).value_counts())
print()
print(df['category'].value_counts())


category
<class 'str'>    1310
Name: count, dtype: int64

category
Food Delivery        344
Transport            250
Quick Commerce       157
E-commerce           127
Cafe                  99
Restaurants           73
Utilities             61
Personal Transfer     45
Subscriptions         35
Fuel                  28
Investments           23
Groceries             22
Cash Withdrawal       17
Entertainment         13
Uncategorised         10
Income                 6
Name: count, dtype: int64


# FEATURE 4: SPENDING OVERVIEW

In [54]:
# FEATURE 4: SPENDING OVERVIEW

vendor_spend = real_spend.groupby('vendor_clean')['amount'].sum().sort_values(ascending=False)
vendor_count = real_spend.groupby('vendor_clean')['amount'].count()
debits = df[df['type'] == 'debit']
credits = df[df['type'] == 'credit']

total_debit = debits['amount'].sum()
total_credit = credits['amount'].sum()
net = total_credit - total_debit
savings_rate = (net / total_credit) * 100

real_spend = debits[~debits['category'].isin(
    ['Personal Transfer', 'Cash Withdrawal', 'Income', 'Uncategorised']
)]
real_total = real_spend['amount'].sum()

cat_spend = real_spend.groupby('category')['amount'].sum().sort_values(ascending=False)
cat_pct = (cat_spend / real_total) * 100

print("Category percentages:")
print("----------------------------------------------------------")
for cat, pct in cat_pct.items():
    bar = '#' * int(pct)
    print(f"  {cat:<20} {bar:<25} {pct:.1f}%")

Category percentages:
----------------------------------------------------------
  E-commerce           ############################ 28.4%
  Investments          #################         17.4%
  Utilities            ############              12.0%
  Food Delivery        ##########                10.6%
  Restaurants          ########                  8.3%
  Fuel                 ######                    6.3%
  Quick Commerce       ######                    6.1%
  Transport            ####                      4.0%
  Groceries            ##                        2.6%
  Cafe                 ##                        2.2%
  Subscriptions        #                         1.6%
  Entertainment                                  0.6%


# Feature 5 — Monthly Trends karte hain:

In [59]:
df['month'] = df['date'].dt.month
month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'June'}

print("month column:", df['month'].unique())

real_spend = df[
    (df['type'] == 'debit') &
    (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Income', 'Uncategorised']))
].copy()

print("real_spend columns:", real_spend.columns.tolist())
print("month in real_spend:", 'month' in real_spend.columns)


monthly = real_spend.pivot_table(
    values='amount',
    index='category',
    columns='month',
    aggfunc='sum',
    fill_value=0
)

print("="*40)
print("  MONTHLY TREND — Food Delivery")
print("="*40)

food_monthly = monthly.loc['Food Delivery']
max_val = food_monthly.max()
for month_num, amt in food_monthly.items():
    bar_len = int((amt / max_val) * 20) if max_val > 0 else 0
    bar = '#' * bar_len
    print(f"  {month_names[month_num]}  Rs.{amt:>8,.0f}  {bar}")

print("\n  CATEGORY GROWTH (Jan → Jun):")
print(f"  {'─'*36}")
for cat in monthly.index:
    jan = monthly.loc[cat, 1]
    jun = monthly.loc[cat, 6]
    if jan > 0:
        growth = ((jun - jan) / jan) * 100
        direction = "↑" if growth > 0 else "↓"
        print(f"  {cat:<20} {direction} {abs(growth):.1f}%")



month column: [1 2 3 4 5 6]
real_spend columns: ['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref', 'date', 'amount', 'type', 'vendor_clean', 'category', 'month']
month in real_spend: True
  MONTHLY TREND — Food Delivery
  Jan  Rs.  22,633  ################
  Feb  Rs.  23,740  #################
  Mar  Rs.  24,803  #################
  Apr  Rs.  27,756  ####################
  May  Rs.  25,408  ##################
  June  Rs.  26,499  ###################

  CATEGORY GROWTH (Jan → Jun):
  ────────────────────────────────────
  Cafe                 ↑ 57.2%
  E-commerce           ↑ 12.2%
  Entertainment        ↑ 51.5%
  Food Delivery        ↑ 17.1%
  Fuel                 ↓ 90.5%
  Groceries            ↓ 67.3%
  Investments          ↓ 39.3%
  Quick Commerce       ↓ 25.8%
  Restaurants          ↑ 30.7%
  Subscriptions        ↑ 66.1%
  Transport            ↓ 36.4%
  Utilities            ↑ 14.8%


# FEATURE 6: TIME OF DAY

In [60]:
# FEATURE 6: TIME OF DAY

df['hour'] = df['Time'].str[:2].astype(int)

real_spend = df[
    (df['type'] == 'debit') &
    (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Income', 'Uncategorised']))
].copy()

food_orders = real_spend[real_spend['category'] == 'Food Delivery']
late_night = food_orders[
    (food_orders['hour'] >= 21) | (food_orders['hour'] <= 2)
]
late_pct = len(late_night) / len(food_orders) * 100

print("="*60)
print("                  TIME-OF-DAY PATTERNS")
print("="*60)
print(f"\n  Food Delivery late night (9PM-2AM): {late_pct:.1f}% of orders")
print("\n  PEAK HOURS PER CATEGORY:")
print(f"  {'-'*35}")
for cat in ['Food Delivery', 'Cafe', 'Quick Commerce', 'Transport', 'E-commerce']:
    cat_df = real_spend[real_spend['category'] == cat]
    if len(cat_df) > 0:
        peak = cat_df['hour'].mode()[0]
        print(f"  {cat:<20} Peak at {peak:02d}:00")

# For my Bonus: Day of week analysis
df['day_of_week'] = df['date'].dt.day_name()
weekend_spend = real_spend[
    real_spend['date'].dt.day_name().isin(['Saturday', 'Sunday'])
]['amount'].sum()
weekday_spend = real_spend[
    ~real_spend['date'].dt.day_name().isin(['Saturday', 'Sunday'])
]['amount'].sum()

weekend_per_day = weekend_spend / (6 * 2 * 4)
weekday_per_day = weekday_spend / (6 * 5 * 4)
diff = ((weekend_per_day - weekday_per_day) / weekday_per_day) * 100

print(f"\n  DAY OF WEEK ANALYSIS:")
print(f"  {'-'*30}")
print(f"  Weekend avg/day : Rs. {weekend_per_day:,.0f}")
print(f"  Weekday avg/day : Rs. {weekday_per_day:,.0f}")
print(f"  Weekends are {abs(diff):.0f}% {'more' if diff > 0 else 'less'} expensive")

                  TIME-OF-DAY PATTERNS

  Food Delivery late night (9PM-2AM): 21.2% of orders

  PEAK HOURS PER CATEGORY:
  -----------------------------------
  Food Delivery        Peak at 20:00
  Cafe                 Peak at 10:00
  Quick Commerce       Peak at 21:00
  Transport            Peak at 20:00
  E-commerce           Peak at 05:00

  DAY OF WEEK ANALYSIS:
  ------------------------------
  Weekend avg/day : Rs. 8,934
  Weekday avg/day : Rs. 8,318
  Weekends are 7% more expensive


# FEATURE 7: ANOMALY DETECTION

In [61]:
# FEATURE 7: ANOMALY DETECTION

cat_mean = real_spend.groupby('category')['amount'].transform('mean')
cat_std  = real_spend.groupby('category')['amount'].transform('std')

real_spend = real_spend.copy()
real_spend['z_score'] = (real_spend['amount'] - cat_mean) / cat_std

# # #  Z > 2 = big transition usually
anomalies = real_spend[real_spend['z_score'] > 2].sort_values('z_score', ascending=False)

print("="*60)
print("  ANOMALY DETECTION (z-score > 2)")
print("="*60)
print(f"\n  Total anomalies found: {len(anomalies)}")
print(f"\n  {'Date':<12} {'Vendor':<18} {'Category':<16} {'Amount':>9}  Z-Score")
print(f"  {'-'*68}")

for _, row in anomalies.head(10).iterrows():
    date_str = str(row['date'])[:10]
    print(f"  {date_str:<12} {row['vendor_clean']:<18} "
          f"{row['category']:<16} Rs.{row['amount']:>8,.0f}  z={row['z_score']:.1f}")

  ANOMALY DETECTION (z-score > 2)

  Total anomalies found: 33

  Date         Vendor             Category            Amount  Z-Score
  --------------------------------------------------------------------
  2024-03-12   BigBasket          Quick Commerce   Rs.   1,865  z=4.6
  2024-06-26   Amazon             E-commerce       Rs.  22,008  z=4.5
  2024-01-23   BigBasket          Quick Commerce   Rs.   1,797  z=4.4
  2024-02-26   Restaurant         Restaurants      Rs.   8,383  z=3.9
  2024-06-22   Restaurant         Restaurants      Rs.   7,935  z=3.6
  2024-03-31   Restaurant         Restaurants      Rs.   7,931  z=3.6
  2024-03-02   Amazon             E-commerce       Rs.  18,273  z=3.6
  2024-05-27   Flipkart           E-commerce       Rs.  17,831  z=3.5
  2024-04-18   BigBasket          Quick Commerce   Rs.   1,518  z=3.4
  2024-02-17   BigBasket          Quick Commerce   Rs.   1,517  z=3.4


# FEATURE 8: SPENDING ARCHETYPES

In [62]:
# FEATURE 8: SPENDING ARCHETYPES
real_total = real_spend['amount'].sum()


def get_pct(category_name):
    amt = real_spend[real_spend['category'] == category_name]['amount'].sum()
    return (amt / real_total) * 100

# archtype : foodie
def check_foodie():
    pct = get_pct('Food Delivery') + get_pct('Restaurants') + get_pct('Cafe')
    if pct > 25:
        return f" THE FOODIE ({pct:.1f}% on food)"
    return None
# archtype : the QC junkie
def check_qc_junkie():
    pct = get_pct('Quick Commerce')
    if pct > 15:
        return f" THE QUICK COMMERCE JUNKIE ({pct:.1f}%)"
    return None

# archtype :shopaholic
def check_shopaholic():
    pct = get_pct('E-commerce')
    if pct > 15:
        return f" THE SHOPAHOLIC ({pct:.1f}%)"
    return None

# archtype : investor
def check_investor():
    pct = get_pct('Investments')
    if pct > 15:
        return f" THE INVESTOR ({pct:.1f}%)"
    return None

# archtype : Late night snacker
def check_late_night():
    food_df = real_spend[real_spend['category'] == 'Food Delivery']
    late = food_df[(food_df['hour'] >= 21) | (food_df['hour'] <= 2)]
    pct = len(late) / len(food_df) * 100
    if pct > 50:
        return f" THE LATE-NIGHT SNACKER ({pct:.1f}% orders after 9PM)"
    return None

# archtype : cab commuter
def check_cab_commuter():
    pct = get_pct('Transport')
    if pct > 10:
        return f"🚗 THE CAB COMMUTER ({pct:.1f}%)"
    return None

# archtype : subscription lover
def check_subscription_lover():
    sub_vendors = real_spend[
        real_spend['category'] == 'Subscriptions'
    ]['vendor_clean'].nunique()
    if sub_vendors >= 4:
        return f" THE SUBSCRIPTION LOVER ({sub_vendors} active subs)"
    return None

# archtype : yolo spender
def check_yolo():
    if savings_rate < 10:
        return f" THE YOLO SPENDER (savings rate {savings_rate:.1f}%)"
    return None

# archtype : disciplined saver
def check_disciplined_saver():
    if savings_rate > 40:
        return f"💰 THE DISCIPLINED SAVER ({savings_rate:.1f}% savings)"
    return None

# extra one : morning cafe ( cafe visit btwn 1-11 am)
def check_morning_cafe():
    cafe_df = real_spend[real_spend['category'] == 'Cafe']
    morning = cafe_df[
        (cafe_df['hour'] >= 7) & (cafe_df['hour'] <= 11)
    ]
    pct = len(morning) / len(cafe_df) * 100 if len(cafe_df) > 0 else 0
    if pct > 50:
        return f" THE PAVEMENT COFFEE CONNOISSEUR ({pct:.1f}% cafe visits 7-11AM)"
    return None

archetype_funcs = [
    check_foodie, check_qc_junkie, check_shopaholic,
    check_investor, check_late_night, check_cab_commuter,
    check_subscription_lover, check_yolo,
    check_disciplined_saver, check_morning_cafe
]

archetypes_found = []
for func in archetype_funcs:
    result = func()
    if result:
        archetypes_found.append(result)

print("="*60)
print("               RAHUL'S SPENDING ARCHETYPES")
print("="*60)
for a in archetypes_found:
    print(f"  -> {a}")
print(f"\n  Total archetypes matched: {len(archetypes_found)}")

               RAHUL'S SPENDING ARCHETYPES
  ->  THE SHOPAHOLIC (28.4%)
  ->  THE INVESTOR (17.4%)
  ->  THE SUBSCRIPTION LOVER (4 active subs)
  ->  THE YOLO SPENDER (savings rate -229.3%)

  Total archetypes matched: 4


# FINAL REPORT

In [69]:
# FINAL REPORT

print("\n" + "="*60)
print(f"{'SpendDNA REPORT — RAHUL SHARMA':^60}")
print(f"{'6 months | 1,310 transactions | Jan–Jun 2024':^60}")
print("="*60)

print(f"""
  EXECUTIVE SUMMARY
  {'─'*50}
  Total Credits  : Rs. {total_credit:>10,.0f}
  Total Debits   : Rs. {total_debit:>10,.0f}
  Net Change     : Rs. {net:>10,.0f} ({'SAVING' if net>=0 else 'OVERSPENDING'})
  Savings Rate   : {savings_rate:.1f}% ({'BURNING SAVINGS' if savings_rate < 0 else 'HEALTHY'})
  Transactions   : {len(df)}
  Unique Vendors : {df['vendor_clean'].nunique()}
""")

print(f"  TOP CATEGORIES (% of real spend)")
print(f"  {'─'*70}")
for cat, amt in cat_spend.head(6).items():
    pct = cat_pct[cat]
    bar = '#' * int(pct)
    print(f"  {cat:<20} {bar:<25} {pct:>5.1f}%  Rs.{amt:>9,.0f}")

print(f"\n  TOP VENDORS")
print(f"  {'─'*50}")
for vendor, amt in vendor_spend.head(5).items():
    count = vendor_count[vendor]
    print(f"  {vendor:<18} Rs.{amt:>9,.0f}  ({count:>3} orders)")

print(f"\n  TIME-OF-DAY")
print(f"  {'─'*50}")
print(f"  Food Delivery late night (9PM-2AM) : {late_pct:.1f}%")
print(f"  Cafe peak                          : 10:00 AM")
print(f"  Weekend vs Weekday                 : Weekends {abs(diff):.0f}% {'more' if diff > 0 else 'less'} expensive")

print(f"\n  MONTHLY TREND — Food Delivery")
print(f"  {'─'*50}")
food_monthly = monthly.loc['Food Delivery']
max_val = food_monthly.max()
for month_num, amt in food_monthly.items():
    bar_len = int((amt / max_val) * 20) if max_val > 0 else 0
    bar = '#' * bar_len
    print(f"  {month_names[month_num]}  Rs.{amt:>8,.0f}  {bar}")

print(f"\n  TOP ANOMALIES")
print(f"  {'─'*50}")
for _, row in anomalies.head(5).iterrows():
    date_str = str(row['date'])[:10]
    print(f"  {date_str}  {row['vendor_clean']:<15} Rs.{row['amount']:>8,.0f}  z={row['z_score']:.1f}")

print(f"\n  SPENDING ARCHETYPES")
print(f"  {'─'*50}")
for a in archetypes_found:
    print(f"  {a}")

print("\n" + "="*63)
print("                      KEY INSIGHTS")
print("="*63)
monthly_burn = abs(net) / 6
print(f"""
  1. Rahul is burning Rs.{monthly_burn:,.0f} per month from savings.
     At -229.3% savings rate, this is completely unsustainable
     and savings will run out very soon.

  2. E-commerce dominates at {cat_pct['E-commerce']:.1f}% of total spend —
     Amazon, Flipkart and Myntra together account for
     Rs.{cat_spend['E-commerce']:,.0f} over 6 months.

  3. Investments at {cat_pct['Investments']:.1f}% show financial awareness,
     but discretionary spending (E-com + Food + Q-com =
     {cat_pct['E-commerce']+cat_pct['Food Delivery']+cat_pct['Quick Commerce']:.0f}%)
     completely overshadows savings behaviour.
""")
print("="*63)


               SpendDNA REPORT — RAHUL SHARMA               
        6 months | 1,310 transactions | Jan–Jun 2024        

  EXECUTIVE SUMMARY
  ──────────────────────────────────────────────────
  Total Credits  : Rs.    509,774
  Total Debits   : Rs.  1,678,901
  Net Change     : Rs. -1,169,127 (OVERSPENDING)
  Savings Rate   : -229.3% (BURNING SAVINGS)
  Transactions   : 1310
  Unique Vendors : 37

  TOP CATEGORIES (% of real spend)
  ──────────────────────────────────────────────────────────────────────
  E-commerce           ############################  28.4%  Rs.  404,997
  Investments          #################          17.4%  Rs.  248,160
  Utilities            ############               12.0%  Rs.  171,911
  Food Delivery        ##########                 10.6%  Rs.  150,839
  Restaurants          ########                    8.3%  Rs.  117,737
  Fuel                 ######                      6.3%  Rs.   89,303

  TOP VENDORS
  ──────────────────────────────────────────────